In [ ]:
# [D0a] Setup (plotting only)
import os, numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

OUT_DS = "/kaggle/working/dataset_figures"; os.makedirs(OUT_DS, exist_ok=True)
CLASSES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]; K = len(CLASSES)
CCOL = plt.get_cmap("tab10")
C_INT, C_EXT = "#3b6ea5", "#e08a3c"
mpl.rcParams.update({"font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 120,
                     "savefig.dpi": 300, "savefig.bbox": "tight", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

def save_ds(fig, name):
    fig.savefig(f"{OUT_DS}/{name}.png"); fig.savefig(f"{OUT_DS}/{name}.pdf"); plt.show(); print("saved", name)


In [ ]:
# [D0b] Load datasets and rebuild the SAME split as your training (seed 42)
import torch
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE = "/kaggle/input/datasets/koushgh45/colorectal-cancer-histopathology"
NCT = f"{BASE}/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
EXT = f"{BASE}/CRC-VAL-HE-7K/CRC-VAL-HE-7K"
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])   # raw [0,1] pixels, no normalisation
full, ext = datasets.ImageFolder(NCT, tf), datasets.ImageFolder(EXT, tf)
assert full.classes == CLASSES and ext.classes == CLASSES, (full.classes, ext.classes)

n = len(full); ntr, nva = int(0.70 * n), int(0.15 * n); nte = n - ntr - nva
tr_ds, va_ds, te_ds = random_split(full, [ntr, nva, nte], generator=torch.Generator().manual_seed(42))
y_full = np.array(full.targets); y_int = y_full[te_ds.indices]; y_ext = np.array(ext.targets)
count_full  = np.bincount(y_full, minlength=K)
count_train = np.bincount(y_full[tr_ds.indices], minlength=K)
count_val   = np.bincount(y_full[va_ds.indices], minlength=K)
count_test  = np.bincount(y_int, minlength=K)
count_ext   = np.bincount(y_ext, minlength=K)
print("full", count_full.sum(), "| train", count_train.sum(), "| val", count_val.sum(), "| test", count_test.sum(), "| external", count_ext.sum())


In [ ]:
# [D1] Figure d1 - class counts (100K by split, and the external set)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.5, 1]}); x = np.arange(K)
b = np.zeros(K)
for nm, cnt, c in [("train (70%)", count_train, "#3b6ea5"), ("val (15%)", count_val, "#8fb1d6"), ("test (15%)", count_test, "#c7d8ea")]:
    axes[0].bar(x, cnt, bottom=b, label=nm, color=c); b += cnt
for i in range(K): axes[0].text(i, b[i] + 100, f"{int(count_full[i]):,}", ha="center", fontsize=8)
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASSES); axes[0].set_title(f"NCT-CRC-HE-100K by split (total {int(count_full.sum()):,})")
axes[0].legend(frameon=False, ncol=3, loc="upper center"); axes[0].set_ylim(0, count_full.max() * 1.22); axes[0].set_ylabel("Images")
axes[1].bar(x, count_ext, color=C_EXT)
for i in range(K): axes[1].text(i, count_ext[i] + 15, f"{int(count_ext[i]):,}", ha="center", fontsize=8)
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASSES, fontsize=8); axes[1].set_title(f"CRC-VAL-HE-7K (total {int(count_ext.sum()):,})"); axes[1].set_ylim(0, count_ext.max() * 1.15)
fig.tight_layout(); save_ds(fig, "d1_class_counts")


In [ ]:
# [D2] Image statistics (mean RGB, saturation, optical density) + sample images. ~2-6 min
N_STATS = 4500

@torch.no_grad()
def image_stats(ds, idx):
    rgb, sat, od, lab = [], [], [], []
    for xb, yb in DataLoader(Subset(ds, list(map(int, idx))), batch_size=128, num_workers=4):
        xb = xb.to(device); mx, mn = xb.max(1).values, xb.min(1).values
        rgb.append(xb.mean((2, 3)).cpu()); sat.append(((mx - mn) / (mx + 1e-6)).mean((1, 2)).cpu())
        od.append((-torch.log10((xb * 255 + 1) / 256)).mean((2, 3)).cpu()); lab.append(yb)
    return dict(rgb=torch.cat(rgb).numpy(), sat=torch.cat(sat).numpy(), od=torch.cat(od).numpy(), labels=torch.cat(lab).numpy())

stats_int = image_stats(te_ds, np.random.RandomState(2).choice(len(te_ds), min(N_STATS, len(te_ds)), replace=False))
stats_ext = image_stats(ext,   np.random.RandomState(2).choice(len(ext),   min(N_STATS, len(ext)),   replace=False))

def make_grid(ds, labels, per_class=4, seed=3):
    g = np.zeros((K, per_class, 224, 224, 3), np.uint8); r = np.random.RandomState(seed)
    for c in range(K):
        for j, i in enumerate(r.choice(np.where(labels == c)[0], per_class, replace=False)):
            g[c, j] = (ds[int(i)][0].permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
    return g
grid_int, grid_ext = make_grid(te_ds, y_int), make_grid(ext, y_ext)
print("stats + sample images ready")


In [ ]:
# [D3] Figure d2 - sample patches per class, internal vs external
fig, axes = plt.subplots(6, K, figsize=(1.6 * K, 1.65 * 6))
for r in range(6):
    g, j = (grid_int, r) if r < 3 else (grid_ext, r - 3)
    for c in range(K):
        axes[r, c].imshow(g[c, j]); axes[r, c].axis("off")
        if r == 0: axes[r, c].set_title(CLASSES[c], fontsize=10)
fig.text(0.005, 0.75, "Internal\n(NCT-CRC-HE-100K)", rotation=90, va="center", fontsize=10, fontweight="bold")
fig.text(0.005, 0.27, "External\n(CRC-VAL-HE-7K)", rotation=90, va="center", fontsize=10, fontweight="bold")
fig.tight_layout(rect=(0.02, 0, 1, 1)); save_ds(fig, "d2_sample_grid")


In [ ]:
# [D4] Figure d3 - colour histograms, internal vs external
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
feats = [("Mean R", lambda d: d["rgb"][:, 0]), ("Mean G", lambda d: d["rgb"][:, 1]),
         ("Mean B", lambda d: d["rgb"][:, 2]), ("Mean saturation", lambda d: d["sat"])]
for ax, (nm, fn) in zip(axes, feats):
    for d, c, lb in [(stats_int, C_INT, "Internal"), (stats_ext, C_EXT, "External")]:
        v = fn(d); ax.hist(v, bins=40, density=True, histtype="stepfilled", alpha=.45, color=c, label=lb)
        ax.hist(v, bins=40, density=True, histtype="step", color=c, lw=1.5)
    ax.set_title(nm)
axes[0].legend(frameon=False); fig.suptitle("Image-level colour statistics: internal vs external", fontweight="bold")
fig.tight_layout(); save_ds(fig, "d3_colour_histograms")


In [ ]:
# [D5] Figure d4 - stain-space (optical density) shift, per class
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharex=True, sharey=True)
for d, c, lb in [(stats_int, C_INT, "Internal"), (stats_ext, C_EXT, "External")]:
    axes[0].scatter(d["od"][:, 0], d["od"][:, 1], s=3, alpha=.25, color=c, label=lb)
axes[0].set_xlabel("Mean optical density, R channel (hematoxylin-weighted)"); axes[0].set_ylabel("Mean optical density, G channel (eosin-weighted)")
axes[0].legend(markerscale=5, frameon=False); axes[0].set_title("All sampled images")
for k in range(K):
    a = stats_int["od"][stats_int["labels"] == k][:, :2].mean(0); b_ = stats_ext["od"][stats_ext["labels"] == k][:, :2].mean(0)
    axes[1].annotate("", xy=b_, xytext=a, arrowprops=dict(arrowstyle="->", color=CCOL(k), lw=2))
    axes[1].scatter(*a, color=CCOL(k), s=30); axes[1].text(*b_, CLASSES[k], fontsize=9, color=CCOL(k), fontweight="bold")
axes[1].set_xlabel("Mean OD, R"); axes[1].set_title("Class centroids: internal (dot) → external (arrow head)")
fig.tight_layout(); save_ds(fig, "d4_stain_space_shift")


In [ ]:
# [D6] Figure d5 - average colour of each class, internal vs external, and how far it moved
rgb_i = np.array([stats_int["rgb"][stats_int["labels"] == k].mean(0) for k in range(K)])
rgb_e = np.array([stats_ext["rgb"][stats_ext["labels"] == k].mean(0) for k in range(K)])
dist = np.linalg.norm((rgb_e - rgb_i) * 255, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axes[0]; ax.set_xlim(0, K); ax.set_ylim(0, 2); ax.grid(False)
for k in range(K):
    ax.add_patch(Rectangle((k, 1), 1, 1, color=np.clip(rgb_i[k], 0, 1))); ax.add_patch(Rectangle((k, 0), 1, 1, color=np.clip(rgb_e[k], 0, 1)))
ax.set_xticks(np.arange(K) + .5); ax.set_xticklabels(CLASSES); ax.set_yticks([1.5, .5]); ax.set_yticklabels(["Internal", "External"])
ax.set_title("Mean colour of each class")
for sp in ax.spines.values(): sp.set_visible(False)
axes[1].bar(CLASSES, dist, color=[CCOL(k) for k in range(K)]); axes[1].set_ylabel("Colour shift (RGB units, 0-255)"); axes[1].set_title("Internal → external colour shift")
for i, v in enumerate(dist): axes[1].text(i, v + .3, f"{v:.0f}", ha="center", fontsize=8)
fig.tight_layout(); save_ds(fig, "d5_class_colour_shift")
